In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from keras.models import load_model
from collections import deque
import winsound
import time



In [2]:
model = load_model("drowsiness_lstm_model.h5")

In [ ]:
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1)
mp_draw = mp.solutions.drawing_utils 


In [4]:
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH = [61, 291, 81, 178, 13, 14, 311, 402]


In [5]:
SEQ_LEN = 30
sequence = deque(maxlen=SEQ_LEN)

In [6]:
def euclidean(p1, p2):
    return np.linalg.norm(p1 - p2)

def eye_ratio(eye):
    return (euclidean(eye[1], eye[5]) + euclidean(eye[2], eye[4])) / (2.0 * euclidean(eye[0], eye[3]))

def mouth_ratio(mouth):
    return (euclidean(mouth[2], mouth[6]) + euclidean(mouth[3], mouth[5])) / (2.0 * euclidean(mouth[0], mouth[4]))


In [12]:
last_beep_time = 0
beep_INTERVAL = 3  

drowsy_time_limit= 3  
drowsy_start_time = None

sequence = []

status = "ALERT"
color = (0, 255, 0)

video_path = "drowsy_driving.mp4"   
cap = cv2.VideoCapture(video_path)


while cap.isOpened():
    success, frame = cap.read()
    if not success:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue

    frame = cv2.resize(frame, (640,480))
    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    current_time = time.time()

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:

            h, w, _ = frame.shape
            landmarks = np.array([(int(lm.x * w), int(lm.y * h))for lm in face_landmarks.landmark])

            left_eye = landmarks[LEFT_EYE]
            right_eye = landmarks[RIGHT_EYE]
            mouth = landmarks[MOUTH]

            ear = (eye_ratio(left_eye) + eye_ratio(right_eye)) / 2.0
            mar = mouth_ratio(mouth)

          
            sequence.append([ear, mar])   

            
            if len(sequence) > SEQ_LEN:   
                sequence.pop(0)

            
            pred = 0.0
            if len(sequence) == SEQ_LEN:
                X = np.array(sequence).reshape(1, SEQ_LEN, 2)
                pred = float(model.predict(X, verbose=0)[0][0])

            if pred > 0.6:
                status = "DROWSY"
                color = (0, 0, 255)

                if drowsy_start_time is None:
                    drowsy_start_time = current_time

                else:
                    drowsy_duration = current_time - drowsy_start_time

                    #alarm 
                    if drowsy_duration >= 10:
                        winsound.Beep(3500,500)  
                        alert_text = "DANGER !!!"
                    elif drowsy_duration >= 6:
                        winsound.Beep(2500,200)   
                        alert_text = "WAKE UP !!!"
                    elif drowsy_duration >= 3:
                        winsound.Beep(1500,120)  
                        alert_text = "ALERT !!!"
                    else:
                        alert_text = None

                    if alert_text:
                        cv2.putText(frame, alert_text, (30, 120),cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

            else:
                status = "ALERT"
                color = (0, 255, 0)
                drowsy_start_time = None


    
    cv2.putText(frame, status, (30, 60),cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

    cv2.imshow("Driver Drowsiness Detection", frame)
    if cv2.waitKey(20) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
